# Lesson 03 - Corak Reka Bentuk Agenik

Dalam pelajaran ini, kita meneroka tiga corak reka bentuk asas untuk membina ejen AI yang berkesan:

1. **Arahan Agen Jelas** — Menyusun isyarat yang tepat dan menetapkan peranan yang membimbing tingkah laku ejen  
2. **Output Berstruktur dengan Model Pydantic** — Memastikan ejen mengembalikan data yang boleh diramal dan disahkan  
3. **Ejen Tanggungjawab Tunggal** — Mereka bentuk ejen fokus yang setiap satunya melakukan satu perkara dengan baik

Kita akan menerapkan setiap corak kepada senario **pengesyor destinasi perjalanan**, secara berperingkat membina sistem yang boleh mencadangkan destinasi, memeriksa ketersediaan, dan mengendalikan logistik.


## Persediaan


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity pydantic --quiet

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated
from pydantic import BaseModel
from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

## Corak 1: Arahan Agen yang Jelas

Corak yang paling berkesan juga yang paling mudah: menulis arahan yang jelas dan terperinci untuk ejen anda.

Arahan yang baik mentakrifkan:
- **Siapa** ejen itu (persona dan nada)
- **Apa** yang perlu dilakukan (tanggungjawab langkah demi langkah)
- **Bagaimana** ia perlu berkelakuan (sekatan dan gaya)

Di bawah, kami mencipta ejen konsierge perjalanan dengan arahan yang jelas yang membentuk setiap respons yang dihasilkannya.


In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

agent = await provider.create_agent(
    name="TravelConcierge",
    instructions="""You are a luxury travel concierge named Alex. Your role is to:
1. Understand the traveler's preferences (budget, climate, activities)
2. Check destination availability before making recommendations
3. Provide detailed, personalized travel suggestions
4. Always mention visa requirements and best travel seasons
Be warm, professional, and enthusiastic about travel.""",
)

response = await agent.run(
    "I'd love a week-long vacation somewhere with great food and history. Budget around $2500."
)
print(response)

## Corak 2: Output Berstruktur dengan Model Pydantic

Teks bebas berguna untuk perbualan, tetapi sistem hiliran memerlukan data berstruktur.  
Dengan menggabungkan **model Pydantic** dengan **fungsi alat**, kita boleh:

- Mendefinisikan skema tepat untuk output ejen  
- Memeriksa kesahihan respons secara automatik  
- Mengintegrasikan keputusan ejen ke dalam logik aplikasi dengan boleh dipercayai  

Kami juga memperkenalkan alat yang mengembalikan butiran destinasi supaya ejen mengasaskan cadangannya pada data sebenar.


In [ ]:
class DestinationRecommendation(BaseModel):
    destination: str
    available: bool
    best_season: str
    highlights: list[str]
    estimated_budget_usd: int


class TravelRecommendations(BaseModel):
    recommendations: list[DestinationRecommendation]
    personalized_note: str


@tool(approval_mode="never_require")
def get_destination_details(destination: Annotated[str, "The destination to look up"]) -> str:
    """Get details about a vacation destination."""
    details = {
        "Barcelona": "Available. Best: May-Jun. Beach, architecture, nightlife. ~$2000/week",
        "Tokyo": "Available. Best: Mar-Apr. Culture, food, technology. ~$2500/week",
        "Cape Town": "Not available. Best: Nov-Mar. Nature, wine, adventure. ~$1800/week",
    }
    return details.get(destination, f"{destination}: No information available.")


structured_agent = await provider.create_agent(
    name="StructuredTravelExpert",
    instructions="You are a travel expert. Recommend destinations based on traveler preferences. Use the get_destination_details tool.",
    tools=[get_destination_details],
)

response = await structured_agent.run(
    "Recommend 3 destinations for a culture-loving traveler with a $2500 budget"
)

if response:
    print(response)

## Corak 3: Ejen Tanggungjawab Tunggal

Tugas yang kompleks mendapat manfaat daripada membahagikan kerja kepada beberapa ejen fokus, masing-masing dengan satu tanggungjawab:

- Seorang **Pakar Destinasi** yang mengetahui tentang tempat dan ketersediaan
- Seorang **Perancang Logistik** yang menguruskan penerbangan, hotel, dan jadual perjalanan

Ini mencerminkan prinsip kejuruteraan perisian *pemisahan kebimbangan* — setiap ejen lebih mudah untuk diuji, diselenggara, dan diperbaiki secara berdikari.


In [ ]:
destination_agent = await provider.create_agent(
    name="DestinationExpert",
    tools=[get_destination_details],
    instructions="""You are a destination research specialist. Your only job is to:
1. Evaluate destinations based on traveler preferences
2. Check availability using the provided tool
3. Return a short ranked list with pros/cons
Do NOT discuss flights, hotels, or logistics — another agent handles that.""",
)

logistics_agent = await provider.create_agent(
    name="LogisticsPlanner",
    instructions="""You are a travel logistics planner. Your only job is to:
1. Create a day-by-day itinerary for the chosen destination
2. Suggest flight and hotel options within the stated budget
3. Note visa requirements and travel insurance recommendations
Do NOT recommend destinations — another agent handles that.""",
)

# Step 1: Destination Expert picks the best options
dest_response = await destination_agent.run(
    "I want a week of culture and food for under $2500. Where should I go?"
)
print("=== Destination Expert ===")
print(dest_response)

# Step 2: Logistics Planner builds the trip plan
logistics_response = await logistics_agent.run(
    f"Plan a week-long trip based on this recommendation:\n{dest_response}"
)
print("\n=== Logistics Planner ===")
print(logistics_response)

## Summary

Dalam pelajaran ini kami menerapkan tiga corak reka bentuk agen pada senario cadangan perjalanan:

| Pattern | Key Idea | Benefit |
|---|---|---|
| **Clear Instructions** | Tentukan persona, tanggungjawab, dan sekatan dari awal | Tingkah laku agen yang konsisten dan selaras jenama |
| **Structured Output** | Gunakan model Pydantic sebagai format respons | Keputusan yang disahkan dan boleh dibaca mesin |
| **Single Responsibility** | Beri setiap agen satu tugas fokus | Lebih mudah untuk diuji, diselenggara, dan digabungkan |

Corak-corak ini berkomposisi secara semula jadi — anda boleh menggabungkan arahan yang jelas dengan output berstruktur dalam agen tanggungjawab tunggal untuk membina sistem yang kukuh dan sedia untuk produksi.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Penafian**:  
Dokumen ini telah diterjemahkan menggunakan perkhidmatan terjemahan AI [Co-op Translator](https://github.com/Azure/co-op-translator). Walaupun kami berusaha untuk ketepatan, sila ambil maklum bahawa terjemahan automatik mungkin mengandungi kesilapan atau ketidaktepatan. Dokumen asal dalam bahasa asalnya harus dianggap sebagai sumber yang sahih. Untuk maklumat yang kritikal, penerjemahan profesional oleh manusia adalah disarankan. Kami tidak bertanggungjawab atas sebarang salah faham atau salah tafsir yang timbul daripada penggunaan terjemahan ini.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
